# LungXplain — ACC-ECBM trên Kaggle

Notebook này tự định nghĩa toàn bộ mô hình và quy trình huấn luyện trong các cell.
Chỉ cần thêm dataset **private** `ehehee/lungxplain-v5-private-input`, chọn GPU T4
và chạy từ trên xuống. Không cần repository hay gói source của LungXplain.

**Luồng xử lý:** ảnh → DenseNet-121 → 14 xác suất concept → Platt → ACC →
nối 22 clinical và 22 cờ thiếu → Explainable Boosting Machine → nguy cơ tử vong nội viện.

Huấn luyện DenseNet riêng cho ba fold ngoài mẫu và một lần refit cuối mỗi seed.
EBM, calibration và ACC chỉ học từ train; validation dùng chọn ngưỡng và báo cáo.
**Test được giữ kín trong notebook này.** Kết quả validation không phải kết quả test.
Nhãn can thiệp được mô phỏng từ báo cáo, không phải xác nhận của bác sĩ.

Mọi dữ liệu và output MIMIC phải nằm trong notebook/account private.

## 1. Thư viện và cấu hình

Mặc định chạy đủ seed 42, 123, 2026. Để chạy thử một lượt thực nghiệm, đổi `seeds`
thành `[42]` trước khi chạy. Epoch tối đa 60, tối thiểu 10, patience 10;
checkpoint chọn bằng AUROC concept trên phần validation nội bộ của train.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    interpret_version = importlib.metadata.version("interpret-core")
except importlib.metadata.PackageNotFoundError:
    interpret_version = None
if interpret_version != "0.7.8":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "interpret-core==0.7.8"])

import gc
import hashlib
import json
import math
import os
import random
import tarfile
import time
import types
import zipfile
from pathlib import Path, PurePosixPath

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             brier_score_loss, log_loss)
from sklearn.model_selection import StratifiedKFold, train_test_split
from interpret.glassbox import ExplainableBoostingClassifier

CFG = {
    "protocol": "lungxplain-v5-native-1",
    "seeds": [42, 123, 2026],
    "outer_folds": 3, "inner_fraction": 0.10,
    "max_epochs": 60, "min_epochs": 10, "patience": 10,
    "batch_size": 16, "num_workers": 2, "image_size": 224,
    "backbone_lr": 1e-4, "head_lr": 3e-4, "weight_decay": 1e-5,
    "gradient_clip": 5.0, "pos_weight_cap": 10.0,
    "alpha_candidates": [0.10, 0.15, 0.20],
    "intervention_fraction": 0.5,
    "bootstrap_repeats": 2000,
    "ebm": {"interactions": 10, "max_rounds": 1500, "outer_bags": 4,
            "max_bins": 64, "max_interaction_bins": 16,
            "early_stopping_rounds": 50, "n_jobs": 4},
}
INPUT_ROOT = Path(os.environ.get("LX_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("LX_WORK_ROOT", "/kaggle/working/lungxplain_native"))
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def plain(value):
    if isinstance(value, dict):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(v) for v in value]
    if isinstance(value, np.ndarray):
        return plain(value.tolist())
    if isinstance(value, np.generic):
        return plain(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(plain(value), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)

ENVIRONMENT = {name: importlib.metadata.version(name) for name in
               ["torch", "torchvision", "numpy", "pandas", "scikit-learn", "interpret-core"]}
print({"device": str(DEVICE), "seeds": CFG["seeds"], "versions": ENVIRONMENT})

## 2. Nạp dữ liệu private và kiểm tra toàn vẹn

Chỉ giải nén **data**, kiểm SHA-256 của archive, schema, manifest và từng ảnh cache.
Ảnh cache là pixel RGB đã resize cạnh ngắn 256, chưa qua mô hình.
Notebook bỏ qua hoàn toàn file ZIP source trong dataset cũ.

In [ ]:
EXPECTED_BUNDLE_SHA = "8b6d121c4f867c4598c012bb1f9fee43e1411247f946e0c1efd8cb7fcd143b4a"
EXPECTED_MANIFEST_SHA = "ac3a6119f4267b31b874c2198990d33593bd6ffbcd1aae4943733bddcc46899a"

def unpack_private_data(input_root, work):
    matches = sorted(input_root.rglob("bundle_manifest_v5.json"))
    matches = [p for p in matches if sha256(p) == EXPECTED_BUNDLE_SHA]
    if len(matches) != 1:
        raise RuntimeError("Attach the audited private v5 input dataset exactly once.")
    bundle_path = matches[0]
    bundle = json.loads(bundle_path.read_text())
    if bundle.get("format") != "lungxplain-private-v5" or bundle.get("private_required") is not True:
        raise RuntimeError("Unexpected dataset bundle.")
    source = bundle_path.parent
    destination = work / "input"
    destination.mkdir(parents=True, exist_ok=True)
    required = {name: digest for name, digest in bundle["members"].items() if name.startswith("data/")}
    aliases = bundle.get("aliases", {})
    for name in required:
        parts = PurePosixPath(name)
        if parts.is_absolute() or ".." in parts.parts or "\\" in name:
            raise RuntimeError("Unsafe data member path.")
    # A re-run reuses verified files; it never relies on a partial extraction.
    missing = [name for name, digest in required.items()
               if not (destination / name).is_file() or sha256(destination / name) != digest]
    archive_candidates = list(source.rglob("lungxplain_v5_data.tar"))
    if missing and archive_candidates:
        if len(archive_candidates) != 1 or sha256(archive_candidates[0]) != bundle["data_tar_sha256"]:
            raise RuntimeError("Data archive checksum mismatch.")
        storage_names = {aliases.get(name, name): name for name in required}
        with tarfile.open(archive_candidates[0], "r:") as archive:
            seen = set()
            for member in archive:
                storage_name = member.name.removeprefix("./")
                canonical = storage_names.get(storage_name)
                if member.isdir():
                    continue
                if canonical is None or not member.isfile() or canonical in seen:
                    raise RuntimeError("Unexpected or duplicate data archive member.")
                seen.add(canonical)
                target = destination / canonical
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.extractfile(member) as src, target.open("wb") as dst:
                    while block := src.read(8 * 1024 * 1024):
                        dst.write(block)
    elif missing:
        # Kaggle may already have expanded the outer archive.
        import shutil
        for name in missing:
            options = [source / aliases.get(name, name), source / name]
            options += list(source.glob("**/" + aliases.get(name, name)))
            candidate = next((p for p in options if p.is_file() and sha256(p) == required[name]), None)
            if candidate is None:
                raise RuntimeError("Missing or altered v5 data file.")
            target = destination / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(candidate, target)
    for name, digest in required.items():
        if not (destination / name).is_file() or sha256(destination / name) != digest:
            raise RuntimeError("Extracted data failed checksum verification.")
    print({"verified_data_files": len(required), "source_code_loaded": False})
    return destination, bundle

DATA_ROOT, BUNDLE = unpack_private_data(INPUT_ROOT, WORK)
DATASET_DIR = DATA_ROOT / BUNDLE["dataset_dir"]
PIXEL_DIR = DATA_ROOT / BUNDLE["pixel_cache"]
MANIFEST = DATASET_DIR / "manifest.csv"
if sha256(MANIFEST) != EXPECTED_MANIFEST_SHA:
    raise RuntimeError("The manifest is not the audited v5 cohort.")
SCHEMA = json.loads((DATASET_DIR / "feature_schema.json").read_text())
VALIDATION = json.loads((DATASET_DIR / "validation_report.json").read_text())
exact = VALIDATION.get("exact_5000_validation", {})
if VALIDATION.get("passed") is not True or exact != {"valid_images": 5000, "required": 5000}:
    raise RuntimeError("A passing exact-5000 dataset validation is required.")
FRAME = pd.read_csv(MANIFEST, low_memory=False)
CONCEPTS = list(SCHEMA["concept_label_columns"])
CLINICAL = list(SCHEMA["model_clinical_columns"])
if len(FRAME) != 5000 or len(CONCEPTS) != 14 or len(CLINICAL) != 44:
    raise RuntimeError("Unexpected cohort dimensions.")
for column in ["subject_id", "hadm_id", "dicom_id"]:
    if FRAME[column].isna().any() or FRAME[column].nunique() != 5000:
        raise RuntimeError("Patient/admission/image uniqueness failed.")
if FRAME["split"].value_counts().to_dict() != {"train": 4000, "valid": 500, "test": 500}:
    raise RuntimeError("Unexpected patient split.")
CACHE_INDEX = json.loads((PIXEL_DIR / "index.json").read_text())
if (CACHE_INDEX["rows"] != 5000 or len(CACHE_INDEX["images"]) != 5000 or
        CACHE_INDEX["configuration"]["manifest_sha256"] != EXPECTED_MANIFEST_SHA):
    raise RuntimeError("Pixel cache row alignment failed.")
IMAGE_PATHS = [PIXEL_DIR / item["file"] for item in CACHE_INDEX["images"]]
for path, entry in zip(IMAGE_PATHS, CACHE_INDEX["images"]):
    if not path.is_file() or sha256(path) != entry["sha256"]:
        raise RuntimeError("Pixel cache integrity failed.")
TRAIN = np.flatnonzero(FRAME["split"].eq("train"))
VALID = np.flatnonzero(FRAME["split"].eq("valid"))
TEST = np.flatnonzero(FRAME["split"].eq("test"))
TARGETS = FRAME[CONCEPTS].to_numpy(dtype=np.float32)
SUPERVISED = (FRAME[["ignore_" + c for c in CONCEPTS]].to_numpy() == 0)
KNOWN = (FRAME[["known_" + c for c in CONCEPTS]].to_numpy() == 1)
if not np.isin(TARGETS, [0, 1]).all():
    raise RuntimeError("Concept targets must be binary.")
# Mortality labels are materialized only for train/validation. Test entries stay NaN.
OUTCOME = np.full(len(FRAME), np.nan)
development = np.r_[TRAIN, VALID]
OUTCOME[development] = FRAME.loc[development, SCHEMA["outcome_column"]].to_numpy(dtype=float)
del development

def clinical_matrix(frame):
    result = frame[CLINICAL].copy()
    for column in CLINICAL:
        if column in SCHEMA["categorical_value_columns"]:
            mapping = {value: index for index, value in enumerate(SCHEMA["categorical_values"][column])}
            if not frame[column].dropna().isin(mapping).all():
                raise ValueError("Unknown clinical category.")
            result[column] = frame[column].map(mapping)
        else:
            result[column] = pd.to_numeric(result[column], errors="raise")
    return result.to_numpy(dtype=np.float64)

Z = clinical_matrix(FRAME)
FEATURE_NAMES = CONCEPTS + CLINICAL
FEATURE_TYPES = ["continuous"] * len(CONCEPTS) + [
    "nominal" if c in SCHEMA["categorical_value_columns"] or c.endswith("_missing") else "continuous"
    for c in CLINICAL]
WEIGHTS_PATH = DATA_ROOT / "data/model_cache/torch/checkpoints/densenet121-a639ec97.pth"
torch.hub.set_dir(str(WEIGHTS_PATH.parent.parent))
print({"patients": len(FRAME), "split": FRAME["split"].value_counts().to_dict(),
       "concepts": len(CONCEPTS), "clinical_inputs": len(CLINICAL), "outcome_inputs": len(FEATURE_NAMES),
       "concept_training": "U-Ignore; unmentioned treated as negative",
       "intervention_references": "explicit positive/negative only", "test_evaluated": False})

## 3. Dataset, DenseNet-121 và loss

Fine-tune toàn bộ CNN với ImageNet initialization. Loss concept bỏ nhãn uncertain;
nhãn không đề cập được xem là âm theo protocol v5. `pos_weight` chỉ tính trên
fitting rows của từng fold, chặn tối đa 10. Mortality không tham gia loss CNN.

In [ ]:
class CXRDataset(Dataset):
    def __init__(self, rows, training=False):
        self.rows = np.asarray(rows, dtype=int)
        steps = []
        if training:
            steps.append(transforms.RandomAffine(5, translate=(0.02, 0.02), scale=(0.98, 1.02)))
        steps += [transforms.CenterCrop(CFG["image_size"]), transforms.ToTensor(),
                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        row = self.rows[item]
        with Image.open(IMAGE_PATHS[row]) as image:
            x = self.transform(image.convert("RGB"))
        return x, torch.from_numpy(TARGETS[row]), torch.from_numpy(SUPERVISED[row].astype(np.float32))

def worker_seed(worker_id):
    value = torch.initial_seed() % 2**32
    np.random.seed(value)
    random.seed(value)

def make_loader(rows, training=False, seed=0):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(CXRDataset(rows, training), batch_size=CFG["batch_size"],
                      shuffle=training, num_workers=CFG["num_workers"],
                      pin_memory=AMP, generator=generator, worker_init_fn=worker_seed)

def build_cnn():
    # Torchvision resolves the bundled ImageNet checkpoint from torch.hub's cache.
    model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    model.classifier = nn.Linear(model.classifier.in_features, len(CONCEPTS))
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    return model.to(DEVICE)

def concept_pos_weight(rows):
    mask = SUPERVISED[rows]
    positives = (TARGETS[rows] * mask).sum(axis=0)
    negatives = ((1 - TARGETS[rows]) * mask).sum(axis=0)
    weights = np.clip(negatives / np.maximum(positives, 1), 1, CFG["pos_weight_cap"])
    return torch.as_tensor(weights, dtype=torch.float32, device=DEVICE)

def concept_loss(logits, labels, mask, pos_weight):
    entries = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight, reduction="none")
    return (entries * mask).sum() / mask.sum().clamp_min(1)

def predict_concepts(model, rows):
    model.eval()
    batches = []
    with torch.inference_mode():
        for x, _, _ in make_loader(rows):
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                probability = torch.sigmoid(model(x.to(DEVICE, non_blocking=True)))
            batches.append(probability.float().cpu().numpy())
    return np.concatenate(batches).astype(np.float64)

def macro_concept_auc(labels, probabilities, mask, required=True):
    scores = []
    for j in range(labels.shape[1]):
        eligible = mask[:, j]
        if np.unique(labels[eligible, j]).size == 2:
            scores.append(roc_auc_score(labels[eligible, j], probabilities[eligible, j]))
    if not scores and required:
        raise ValueError("No two-class concepts for checkpoint selection.")
    return float(np.mean(scores)) if scores else None

## 4. Vòng lặp huấn luyện và checkpoint

Mỗi fold có train/validation nội bộ, độc lập với validation chính và test.
Lưu optimizer, AMP scaler, epoch và lịch sử sau mỗi epoch. Chạy lại notebook
trong cùng working directory sẽ tiếp tục từ epoch đã lưu. Khi mở phiên Kaggle mới,
cần gắn output của phiên trước nếu muốn giữ checkpoint.

In [ ]:
def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    torch.save(value, temporary)
    temporary.replace(path)

def train_stage(folder, fitting_rows, selection_rows, stage_seed, refit_epochs=None):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    fitting_rows = np.asarray(fitting_rows, dtype=int)
    selection_rows = np.asarray(selection_rows, dtype=int)
    if np.intersect1d(fitting_rows, selection_rows).size:
        raise ValueError("Fitting and selection patients overlap.")
    if not np.isin(np.r_[fitting_rows, selection_rows], TRAIN).all():
        raise ValueError("CNN fitting/selection must stay inside train.")
    stage_spec = {"fitting_rows": fitting_rows.tolist(), "selection_rows": selection_rows.tolist(),
                  "stage_seed": stage_seed, "refit_epochs": refit_epochs}
    spec_path = folder / "stage.json"
    if spec_path.exists() and json.loads(spec_path.read_text()) != stage_spec:
        raise RuntimeError("Stage split/configuration changed. Use a new run directory.")
    save_json(spec_path, stage_spec)
    seed_all(stage_seed)
    model = build_cnn()
    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": CFG["backbone_lr"]},
        {"params": model.classifier.parameters(), "lr": CFG["head_lr"]}],
        weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    pos_weight = concept_pos_weight(fitting_rows)
    last_path, best_path = folder / "last.pt", folder / "best.pt"
    history, start, best, best_epoch, stale, steps = [], 0, -math.inf, 0, 0, 0
    done = False
    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        history = checkpoint["history"]
        start, best, best_epoch = checkpoint["epoch"], checkpoint["best"], checkpoint["best_epoch"]
        stale, steps, done = checkpoint["stale"], checkpoint["steps"], checkpoint["done"]
        del checkpoint
    limit = int(refit_epochs if refit_epochs is not None else CFG["max_epochs"])
    for epoch in range(start + 1, limit + 1) if not done else []:
        # Epoch-specific RNG also makes resuming at epoch boundaries reproducible.
        epoch_seed = stage_seed * 1000 + epoch
        seed_all(epoch_seed)
        started = time.monotonic()
        model.train()
        total_loss, total_entries = 0.0, 0
        for x, labels, mask in make_loader(fitting_rows, training=True, seed=epoch_seed):
            x, labels, mask = [v.to(DEVICE, non_blocking=True) for v in (x, labels, mask)]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                loss = concept_loss(model(x), labels, mask, pos_weight)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite CNN loss.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["gradient_clip"])
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            steps += int(scaler.get_scale() >= old_scale)
            count = int(mask.sum().item())
            total_loss += float(loss.detach()) * count
            total_entries += count
        score = None
        improved = False
        if refit_epochs is None:
            p_inner = predict_concepts(model, selection_rows)
            score = macro_concept_auc(TARGETS[selection_rows], p_inner, SUPERVISED[selection_rows])
            improved = score > best + 1e-8
            if improved:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
        else:
            best_epoch = epoch
        done = epoch == limit or (refit_epochs is None and epoch >= CFG["min_epochs"] and stale >= CFG["patience"])
        record = {"epoch": epoch, "concept_loss": total_loss / max(1, total_entries),
                  "inner_concept_auroc": score, "seconds": time.monotonic() - started,
                  "optimizer_steps": steps}
        history.append(record)
        checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                      "scaler": scaler.state_dict(), "epoch": epoch, "best": best,
                      "best_epoch": best_epoch, "stale": stale, "steps": steps,
                      "done": done, "history": history, "pos_weight": pos_weight.cpu()}
        if improved or refit_epochs is not None:
            atomic_torch_save({"model": model.state_dict(), "epoch": epoch}, best_path)
        atomic_torch_save(checkpoint, last_path)
        save_json(folder / "history.json", history)
        print({"stage": folder.name, **plain(record)}, flush=True)
        del checkpoint
    selected = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(selected["model"])
    del selected, optimizer, scaler
    return model, {"best_epoch": best_epoch, "epochs": len(history), "optimizer_steps": steps}

def crossfit_cnn(seed, run):
    cache = run / "concept_predictions.npz"
    if cache.exists():
        with np.load(cache) as saved:
            result = saved["probabilities"].copy()
            if (not np.array_equal(saved["train_rows"], TRAIN) or
                    not np.array_equal(saved["valid_rows"], VALID) or
                    result.shape != (len(FRAME), len(CONCEPTS)) or
                    not np.isfinite(result[np.r_[TRAIN, VALID]]).all() or
                    not np.isnan(result[TEST]).all() or (saved["fold_id"][TRAIN] < 0).any()):
                raise RuntimeError("Cached prediction alignment or test isolation failed.")
            return result
    probabilities = np.full((len(FRAME), len(CONCEPTS)), np.nan, dtype=np.float64)
    fold_id = np.full(len(FRAME), -1, dtype=int)
    selected_epochs, stages = [], []
    outer = StratifiedKFold(n_splits=CFG["outer_folds"], shuffle=True, random_state=seed)
    for fold, (development_positions, heldout_positions) in enumerate(outer.split(TRAIN, OUTCOME[TRAIN])):
        development_rows, heldout_rows = TRAIN[development_positions], TRAIN[heldout_positions]
        fitting_rows, inner_rows = train_test_split(
            development_rows, test_size=CFG["inner_fraction"], random_state=seed + fold,
            stratify=OUTCOME[development_rows])
        if np.intersect1d(heldout_rows, np.r_[fitting_rows, inner_rows]).size:
            raise RuntimeError("Outer fold leakage.")
        model, info = train_stage(run / f"fold{fold}", fitting_rows, inner_rows, seed + fold)
        probabilities[heldout_rows] = predict_concepts(model, heldout_rows)
        fold_id[heldout_rows] = fold
        selected_epochs.append(info["best_epoch"])
        stages.append(info)
        del model
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()
    refit_epochs = max(1, int(np.median(selected_epochs)))
    model, info = train_stage(run / "final_cnn", TRAIN, [], seed + 100, refit_epochs=refit_epochs)
    # Test predictions are not computed here; their cache entries remain NaN.
    probabilities[VALID] = predict_concepts(model, VALID)
    del model
    if not np.isfinite(probabilities[np.r_[TRAIN, VALID]]).all() or np.any(fold_id[TRAIN] < 0):
        raise RuntimeError("Incomplete out-of-fold predictions.")
    temporary = cache.with_name(cache.name + ".partial")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, probabilities=probabilities, fold_id=fold_id,
                            train_rows=TRAIN, valid_rows=VALID)
    temporary.replace(cache)
    save_json(run / "cnn_summary.json", {"folds": stages, "final": info,
                                         "refit_epochs": refit_epochs, "test_predicted": False})
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()
    return probabilities

## 5. Platt calibration và ACC

Platt được fit trên dự đoán ngoài fold của train, riêng từng concept.
ACC dùng ma trận Pearson của nhãn train. Với nhiều xác nhận, cộng đồng thời residual
của từng concept rồi khóa các giá trị đã xác nhận. Không có xác nhận thì giữ nguyên
xác suất đầu vào. Alpha chọn trong `{0.10, 0.15, 0.20}` bằng log-loss concept trên
holdout thuộc train; không dùng mortality hay test để chọn concept/alpha.

In [ ]:
class PlattCalibration:
    def fit(self, probabilities, labels, mask):
        self.models = []
        for j in range(labels.shape[1]):
            eligible = mask[:, j]
            if np.unique(labels[eligible, j]).size != 2:
                self.models.append(None)
                continue
            p = np.clip(probabilities[eligible, j], 1e-6, 1 - 1e-6)
            x = np.log(p / (1 - p)).reshape(-1, 1)
            self.models.append(LogisticRegression(C=1.0, max_iter=1000).fit(x, labels[eligible, j]))
        return self

    def transform(self, probabilities):
        result = np.array(probabilities, dtype=float, copy=True)
        for j, model in enumerate(self.models):
            if model is not None:
                p = np.clip(probabilities[:, j], 1e-6, 1 - 1e-6)
                result[:, j] = model.predict_proba(np.log(p / (1 - p)).reshape(-1, 1))[:, 1]
        return result

class AdaptiveConceptCalibrator:
    def __init__(self, alpha=0.15):
        self.alpha = float(alpha)

    def fit(self, training_labels):
        self.weights = pd.DataFrame(training_labels).corr(method="pearson").fillna(0).to_numpy()
        return self

    def transform(self, probabilities, reference=None, observed=None):
        p = np.array(probabilities, dtype=float, copy=True)
        if p.ndim != 2 or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError("ACC expects a finite probability matrix.")
        if observed is None:
            return p
        observed = np.asarray(observed, dtype=bool)
        if observed.shape != p.shape or reference is None or np.shape(reference) != p.shape:
            raise ValueError("Confirmation shapes must match concept probabilities.")
        reference = np.asarray(reference, dtype=float)
        if not np.isin(reference[observed], [0, 1]).all():
            raise ValueError("Confirmed concepts require binary reference labels.")
        residual = np.where(observed, reference - p, 0.0)
        updated = np.clip(p + self.alpha * (residual @ self.weights), 0.0, 1.0)
        updated[observed] = reference[observed]
        return updated

def hard_correction(probabilities, reference, observed):
    return np.where(observed, reference, probabilities)

def entropy_order(probabilities, known):
    p = np.clip(probabilities, 1e-8, 1 - 1e-8)
    entropy = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-np.where(known, entropy, -np.inf), axis=1, kind="stable")

def confirmation_mask(order, known, count):
    observed = np.zeros_like(known, dtype=bool)
    if count:
        np.put_along_axis(observed, order[:, :count], True, axis=1)
    return observed & known

def concept_logloss_parts(labels, probabilities, mask):
    p = np.clip(probabilities, 1e-7, 1 - 1e-7)
    losses = -(labels * np.log(p) + (1 - labels) * np.log(1 - p))
    return (losses * mask).sum(axis=1), mask.sum(axis=1)

def select_alpha(probabilities, seed):
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(len(TRAIN))
    cut = int(0.8 * len(TRAIN))
    fit_rows, held_rows = TRAIN[shuffled[:cut]], TRAIN[shuffled[cut:]]
    p, labels, known = probabilities[shuffled[cut:]], TARGETS[held_rows], KNOWN[held_rows]
    order = entropy_order(p, known)
    candidates = []
    for alpha in CFG["alpha_candidates"]:
        calibrator = AdaptiveConceptCalibrator(alpha).fit(TARGETS[fit_rows])
        numerator, denominator = 0.0, 0
        for count in [1, 2]:
            observed = confirmation_mask(order, known, count)
            corrected = calibrator.transform(p, labels, observed)
            losses, entries = concept_logloss_parts(labels, corrected, known & ~observed)
            numerator += losses.sum()
            denominator += entries.sum()
        if denominator == 0:
            raise ValueError("No unconfirmed references for alpha selection.")
        candidates.append({"alpha": alpha, "train_holdout_logloss": numerator / denominator})
    best = min(candidates, key=lambda row: row["train_holdout_logloss"])["alpha"]
    return AdaptiveConceptCalibrator(best).fit(TARGETS[TRAIN]), candidates

def intervention_training_inputs(probabilities, calibrator, seed):
    rng = np.random.default_rng(seed)
    observed = np.zeros_like(KNOWN[TRAIN])
    chosen = rng.choice(len(TRAIN), size=int(CFG["intervention_fraction"] * len(TRAIN)), replace=False)
    for index in chosen:
        eligible = np.flatnonzero(KNOWN[TRAIN[index]])
        if len(eligible):
            count = int(rng.integers(1, len(eligible) + 1))
            observed[index, rng.choice(eligible, count, replace=False)] = True
    return calibrator.transform(probabilities, TARGETS[TRAIN], observed), observed

## 6. EBM và các metric

Huấn luyện bốn cấu hình: có/không Platt × EBM chuẩn/intervention-aware.
Mỗi cấu hình chỉ có một EBM; No Correction, Hard Correction và ACC dùng chung EBM đó.
Intervention-aware thay concept của một phần train bằng xác nhận mô phỏng, vẫn một
dòng mỗi bệnh nhân. Ngưỡng Macro-F1 chọn trên validation rồi giữ cố định khi can thiệp.

In [ ]:
def outcome_metrics(labels, probabilities, threshold):
    return {"auroc": roc_auc_score(labels, probabilities) if np.unique(labels).size == 2 else None,
            "auprc": average_precision_score(labels, probabilities) if np.unique(labels).size == 2 else None,
            "macro_f1": f1_score(labels, probabilities >= threshold, average="macro", zero_division=0),
            "brier": brier_score_loss(labels, probabilities),
            "logloss": log_loss(labels, probabilities, labels=[0, 1])}

def select_threshold(labels, probabilities):
    candidates = np.unique(np.r_[0.0, probabilities, np.nextafter(1.0, 2.0)])
    scores = [f1_score(labels, probabilities >= t, average="macro", zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])

def concept_metrics(labels, probabilities, known):
    records = []
    for j, name in enumerate(CONCEPTS):
        eligible = known[:, j]
        if eligible.any():
            y, p = labels[eligible, j], probabilities[eligible, j]
            records.append({"concept": name, "reference_count": int(eligible.sum()),
                            "auroc": roc_auc_score(y, p) if np.unique(y).size == 2 else None,
                            "f1": f1_score(y, p >= 0.5, zero_division=0),
                            "accuracy": float(np.mean(y == (p >= 0.5))),
                            "logloss": log_loss(y, p, labels=[0, 1]),
                            "brier": brier_score_loss(y, p)})
    return records

def fit_ebm(probabilities, rows, seed):
    inputs = np.column_stack([probabilities, Z[rows]])
    if inputs.shape[1] != 58:
        raise ValueError("Outcome head must receive only 14 concepts + 44 clinical columns.")
    ebm = ExplainableBoostingClassifier(feature_names=FEATURE_NAMES, feature_types=FEATURE_TYPES,
                                      random_state=seed, **CFG["ebm"])
    ebm.fit(inputs, OUTCOME[rows].astype(int))
    return ebm

def outcome_probability(ebm, probabilities, rows):
    return ebm.predict_proba(np.column_stack([probabilities, Z[rows]]))[:, 1]

def paired_intervention_ci(labels, clamp_outcome, acc_outcome, clamp_parts, acc_parts, seed):
    rng = np.random.default_rng(seed)
    differences = {"outcome_auroc": [], "outcome_logloss": [], "remaining_concept_logloss": []}
    for _ in range(CFG["bootstrap_repeats"]):
        sampled = rng.integers(0, len(labels), len(labels))
        y = labels[sampled]
        if np.unique(y).size == 2:
            differences["outcome_auroc"].append(
                roc_auc_score(y, acc_outcome[sampled]) - roc_auc_score(y, clamp_outcome[sampled]))
        differences["outcome_logloss"].append(
            log_loss(y, acc_outcome[sampled], labels=[0, 1]) -
            log_loss(y, clamp_outcome[sampled], labels=[0, 1]))
        count = clamp_parts[1][sampled].sum()
        if count:
            differences["remaining_concept_logloss"].append(
                (acc_parts[0][sampled].sum() - clamp_parts[0][sampled].sum()) / count)
    result = {}
    point = {
        "outcome_auroc": roc_auc_score(labels, acc_outcome) - roc_auc_score(labels, clamp_outcome),
        "outcome_logloss": log_loss(labels, acc_outcome, labels=[0, 1]) - log_loss(labels, clamp_outcome, labels=[0, 1]),
        "remaining_concept_logloss": (acc_parts[0].sum() - clamp_parts[0].sum()) / clamp_parts[1].sum()
            if clamp_parts[1].sum() else None}
    for name, values in differences.items():
        result[name] = {"delta_acc_minus_reference": point[name],
                        "ci95": np.quantile(values, [0.025, 0.975]).tolist() if values else None,
                        "valid_resamples": len(values)}
    return result

def evaluate_interventions(ebm, probabilities, calibrator, threshold, seed, folder):
    labels, known = TARGETS[VALID], KNOWN[VALID]
    outcome = OUTCOME[VALID].astype(int)
    order = entropy_order(probabilities, known)
    records, intervals = [], {}
    prediction_cache = {"probabilities": probabilities, "order": order, "known": known}
    for count in range(len(CONCEPTS) + 1):
        observed = confirmation_mask(order, known, count)
        remaining = known & ~observed
        versions = {"no_correction": probabilities,
                    "hard_correction": hard_correction(probabilities, labels, observed),
                    "acc": calibrator.transform(probabilities, labels, observed)}
        outcomes, parts = {}, {}
        for strategy, corrected in versions.items():
            if count == 0 and not np.array_equal(corrected, probabilities):
                raise RuntimeError("No-intervention identity violated.")
            if strategy != "no_correction" and not np.array_equal(corrected[observed], labels[observed]):
                raise RuntimeError("A confirmed concept changed.")
            outcomes[strategy] = outcome_probability(ebm, corrected, VALID)
            parts[strategy] = concept_logloss_parts(labels, corrected, remaining)
            total_count = parts[strategy][1].sum()
            records.append({"requested_concepts": count, "strategy": strategy,
                            "mean_actual_confirmations": float(observed.sum(axis=1).mean()),
                            "remaining_reference_count": int(total_count),
                            "remaining_concept_logloss": parts[strategy][0].sum() / total_count if total_count else None,
                            "remaining_concept_auroc": macro_concept_auc(labels, corrected, remaining, required=False),
                            "remaining_concept_brier": float(((corrected - labels)**2 * remaining).sum() / total_count)
                                if total_count else None,
                            **outcome_metrics(outcome, outcomes[strategy], threshold)})
            prediction_cache[f"{strategy}_m{count}"] = outcomes[strategy]
        if count in [1, 2, 14]:
            intervals[str(count)] = {
                "acc_vs_" + baseline: paired_intervention_ci(
                    outcome, outcomes[baseline], outcomes["acc"], parts[baseline], parts["acc"], seed + count)
                for baseline in ["hard_correction", "no_correction"]}
    pd.DataFrame(records).to_csv(folder / "interventions_validation.csv", index=False)
    np.savez_compressed(folder / "intervention_predictions_validation.npz", **prediction_cache)
    save_json(folder / "paired_bootstrap_validation.json", intervals)
    return records

## 7. Chạy từng seed và lưu kết quả

Mỗi run gắn với hash dữ liệu, code, cấu hình và môi trường. Đổi code/cấu hình thì
dùng thư mục run mới; không tiếp tục nhầm checkpoint cũ. Mọi dự đoán train dùng
để fit EBM đều là ngoài fold. Các cell sau chạy thật khi bấm **Run All**.

In [ ]:
def stable_code(function):
    def encode(value):
        if isinstance(value, types.CodeType):
            return {"bytecode": value.co_code.hex(), "constants": [encode(c) for c in value.co_consts],
                    "names": value.co_names, "variables": value.co_varnames,
                    "freevars": value.co_freevars, "cellvars": value.co_cellvars,
                    "argcount": value.co_argcount, "kwonlyargcount": value.co_kwonlyargcount}
        if isinstance(value, (tuple, list)):
            return [encode(v) for v in value]
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return repr(value)
    return {"body": encode(function.__code__), "defaults": repr(function.__defaults__),
            "kwdefaults": repr(function.__kwdefaults__)}

def implementation_fingerprint():
    definitions = {}
    for name, value in list(globals().items()):
        if isinstance(value, types.FunctionType) and value.__module__ == __name__:
            definitions[name] = stable_code(value)
        elif isinstance(value, type) and value.__module__ == __name__:
            definitions[name] = {method: stable_code(fn) for method, fn in vars(value).items()
                                 if isinstance(fn, types.FunctionType)}
    return hashlib.sha256(json.dumps(definitions, sort_keys=True).encode()).hexdigest()

def bind_run(run, seed):
    binding = {"configuration": CFG, "seed": seed, "code_sha256": implementation_fingerprint(),
               "manifest_sha256": sha256(MANIFEST),
               "schema_sha256": sha256(DATASET_DIR / "feature_schema.json"),
               "cache_index_sha256": sha256(PIXEL_DIR / "index.json"),
               "imagenet_sha256": sha256(WEIGHTS_PATH), "environment": ENVIRONMENT,
               "device": str(DEVICE), "test_evaluated": False}
    run.mkdir(parents=True, exist_ok=True)
    binding_file = run / "binding.json"
    if binding_file.exists() and json.loads(binding_file.read_text()) != binding:
        raise RuntimeError("Run binding changed. Set a new WORK directory before training.")
    save_json(binding_file, binding)

def plot_interventions(records, path):
    table = pd.DataFrame(records)
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    for strategy, label in [("no_correction", "No correction"), ("hard_correction", "Hard correction"), ("acc", "ACC")]:
        selected = table[table.strategy.eq(strategy)]
        for axis, metric, title in zip(axes, ["remaining_concept_logloss", "auroc", "logloss"],
                                       ["Unconfirmed concept log-loss", "Mortality AUROC", "Mortality log-loss"]):
            axis.plot(selected.requested_concepts, selected[metric], marker="o", ms=3, label=label)
            axis.set(xlabel="Requested confirmations (available references only)", title=title)
            axis.grid(alpha=0.2)
    axes[0].legend()
    figure.suptitle("Validation — simulated report-derived concept intervention")
    figure.tight_layout()
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)

def predict_risk(image_path, clinical_values, confirmed_concepts=None, seed=42,
                 variant="calibrated_intervention_aware"):
    run = WORK / "runs" / f"seed{seed}"
    folder = run / variant
    if not (run / "complete.json").is_file():
        raise RuntimeError("Complete this seed's training before inference.")
    seed_all(seed)
    model = build_cnn()
    checkpoint = torch.load(run / "final_cnn/best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    transform = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(CFG["image_size"]), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    with Image.open(image_path) as image:
        x = transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
        probability = torch.sigmoid(model(x)).float().cpu().numpy().astype(float)
    del model, checkpoint
    calibrated_variant = variant.startswith("calibrated_")
    if calibrated_variant:
        probability = joblib.load(run / "platt.joblib").transform(probability)
    prefix = "calibrated" if calibrated_variant else "uncalibrated"
    acc_parameters = json.loads((run / f"{prefix}_acc.json").read_text())
    calibrator = AdaptiveConceptCalibrator(acc_parameters["alpha"])
    calibrator.weights = np.asarray(acc_parameters["weights"])
    observed = np.zeros_like(probability, dtype=bool)
    reference = np.full_like(probability, np.nan)
    for name, value in (confirmed_concepts or {}).items():
        if name not in CONCEPTS or value not in (0, 1):
            raise ValueError("Confirmations must map a known concept name to 0 or 1.")
        index = CONCEPTS.index(name)
        observed[0, index], reference[0, index] = True, value
    corrected = calibrator.transform(probability, reference, observed)
    row = {name: clinical_values.get(name, np.nan) for name in SCHEMA["clinical_value_columns"]}
    row.update({name + "_missing": int(pd.isna(row[name])) for name in SCHEMA["clinical_value_columns"]})
    inputs = np.column_stack([corrected, clinical_matrix(pd.DataFrame([row]))])
    ebm = joblib.load(folder / "ebm.joblib")
    risk = float(ebm.predict_proba(inputs)[0, 1])
    threshold = json.loads((folder / "threshold_validation.json").read_text())["threshold"]
    return {"mortality_probability": risk, "predicted_class": int(risk >= threshold),
            "validation_threshold": threshold,
            "concepts_before_acc": dict(zip(CONCEPTS, probability[0].tolist())),
            "concepts_after_acc": dict(zip(CONCEPTS, corrected[0].tolist())),
            "confirmed_concepts": confirmed_concepts or {},
            "explanation": plain(ebm.explain_local(inputs).data(0))}

def run_seed(seed):
    run = WORK / "runs" / f"seed{seed}"
    bind_run(run, seed)
    complete = run / "complete.json"
    if complete.exists():
        stamp = json.loads(complete.read_text())
        for name, digest in stamp["artifacts"].items():
            if not (run / name).is_file() or sha256(run / name) != digest:
                raise RuntimeError("Completed run artifact changed.")
        return json.loads((run / "validation_summary.json").read_text())
    seed_all(seed)
    raw = crossfit_cnn(seed, run)
    platt = PlattCalibration().fit(raw[TRAIN], TARGETS[TRAIN], SUPERVISED[TRAIN])
    joblib.dump(platt, run / "platt.joblib")
    calibrated = np.full_like(raw, np.nan)
    for rows in [TRAIN, VALID]:
        calibrated[rows] = platt.transform(raw[rows])
    summaries, curves = [], {}
    for use_calibration, probabilities in [(True, calibrated), (False, raw)]:
        calibrator, alpha_selection = select_alpha(probabilities[TRAIN], seed)
        prefix = "calibrated" if use_calibration else "uncalibrated"
        save_json(run / f"{prefix}_acc.json", {"alpha": calibrator.alpha, "weights": calibrator.weights,
                                              "selection": alpha_selection, "selection_split": "train_only"})
        altered, training_mask = intervention_training_inputs(probabilities[TRAIN], calibrator, seed)
        for aware, train_concepts in [(False, probabilities[TRAIN]), (True, altered)]:
            variant = prefix + ("_intervention_aware" if aware else "_standard_ebm")
            folder = run / variant
            folder.mkdir(exist_ok=True)
            ebm = fit_ebm(train_concepts, TRAIN, seed)
            joblib.dump(ebm, folder / "ebm.joblib")
            prediction = outcome_probability(ebm, probabilities[VALID], VALID)
            threshold = select_threshold(OUTCOME[VALID].astype(int), prediction)
            metrics = outcome_metrics(OUTCOME[VALID].astype(int), prediction, threshold)
            summaries.append({"seed": seed, "variant": variant, "threshold": threshold, **metrics})
            save_json(folder / "threshold_validation.json", {"threshold": threshold, "criterion": "macro_f1",
                                                              "selection_split": "valid", "test_evaluated": False})
            np.savez_compressed(folder / "training_intervention_mask.npz",
                                observed=training_mask if aware else np.zeros_like(training_mask))
            np.savez_compressed(folder / "outcome_validation.npz", probabilities=prediction, rows=VALID)
            save_json(folder / "concept_metrics_validation.json",
                      concept_metrics(TARGETS[VALID], probabilities[VALID], KNOWN[VALID]))
            records = evaluate_interventions(ebm, probabilities[VALID], calibrator, threshold, seed, folder)
            curves[variant] = records
            # Global EBM terms, with the 58 human-readable feature names.
            explanation = ebm.explain_global()
            save_json(folder / "global_explanation.json", explanation.data())
            save_json(folder / "global_terms_including_missing_bins.json", {
                "feature_names": ebm.feature_names_in_, "term_features": ebm.term_features_,
                "term_scores": ebm.term_scores_, "intercept": ebm.intercept_, "bins": ebm.bins_})
            print({"seed": seed, "variant": variant, **metrics}, flush=True)
    save_json(run / "validation_summary.json", summaries)
    plot_interventions(curves["calibrated_intervention_aware"], run / "interventions_validation.png")
    artifacts = {str(path.relative_to(run)): sha256(path) for path in sorted(run.rglob("*"))
                 if path.is_file() and path.name != "complete.json" and not path.name.endswith(".partial")}
    save_json(complete, {"seed": seed, "artifacts": artifacts, "test_evaluated": False})
    # One downloadable archive per seed; the notebook/output must remain private.
    archive_path = WORK / f"lungxplain_native_seed{seed}.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(run.rglob("*")):
            if path.is_file() and not path.name.endswith(".partial"):
                archive.write(path, arcname=str(path.relative_to(run.parent)))
    return summaries

In [ ]:
if DEVICE.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU before running the full training cell.")
ALL_RESULTS = []
for seed in CFG["seeds"]:
    print({"starting_seed": seed, "test_evaluated": False}, flush=True)
    ALL_RESULTS.extend(run_seed(seed))
RESULT_TABLE = pd.DataFrame(ALL_RESULTS)
RESULT_TABLE.to_csv(WORK / "validation_all_seeds.csv", index=False)
display(RESULT_TABLE)

## 8. Tổng hợp validation

Mean ± SD chỉ tính khi thực sự có nhiều seed. Đường m=0 là dự đoán tự động chung
cho ba chiến lược; đó không phải ba thành tích độc lập. Ở m=14, số xác nhận thực tế
có thể ít hơn 14 vì chỉ concept có nhãn tường minh mới được dùng làm reference.
Nếu không còn concept chưa xác nhận để chấm, metric tương ứng để trống.

Output gồm checkpoint CNN, OOF predictions, Platt, ma trận ACC/alpha, bốn EBM,
ngưỡng, bảng intervention, paired bootstrap CI và hình. Tất cả nằm trong working
directory và ZIP từng seed. Định nghĩa các class trong notebook này trước khi
load lại model Platt được lưu bằng joblib.

Hàm `predict_risk(image_path, clinical_values, confirmed_concepts, seed)` dùng
checkpoint đã train để dự đoán ảnh mới, trả xác suất concept trước/sau ACC,
nguy cơ mortality và giải thích EBM. `clinical_values` là dictionary với tên
22 biến trong schema; cờ thiếu được tạo tự động. Xác nhận concept là dictionary
`{"Edema": 1, "Pneumothorax": 0}`; để trống nếu chưa can thiệp.

Test chưa được đánh giá. Các baseline clinical-only, black-box và kiểm định
nhạy thời gian thuộc bước so sánh tiếp theo; notebook này huấn luyện ACC-ECBM.

In [ ]:
metrics = ["auroc", "auprc", "macro_f1", "brier", "logloss"]
seed_count = RESULT_TABLE.groupby("variant")["seed"].nunique()
if (seed_count > 1).all():
    display(RESULT_TABLE.groupby("variant")[metrics].agg(["mean", "std"]))
else:
    print("Kết quả một seed; không báo cáo mean ± SD.")
save_json(WORK / "experiment_status.json", {
    "dataset": "v5", "completed_seeds": sorted(RESULT_TABLE.seed.unique().tolist()),
    "evaluation_split": "valid", "test_evaluated": False,
    "source_code_loaded_from_dataset": False, "configuration": CFG})
print("Đã lưu kết quả validation và checkpoint tại:", WORK)